# CELL 0 — Purpose, Data Grain, and Stop Rule

**Task:** Tumor / lesion-mask segmentation (axial CT slices).

**Grain:** One sample = one axial slice identified by `(volume_id, slice_index)`.

**Authoritative source:** Paired official LiTS NIfTI files — `volume-{ID}.nii` and
`segmentation-{ID}.nii` — with the same numeric ID.

**Label convention (NIfTI):**
| Value | Meaning |
|-------|---------|
| 0 | Background |
| 1 | Liver parenchyma |
| 2 | Tumor / lesion |

**Derived mask convention (PNG, 256×256):**
| File | Definition | Mode | Values |
|------|-----------|------|--------|
| `organ_mask` | `segmentation > 0` (liver **including** tumor) | L | {0, 255} |
| `tumor_mask` | `segmentation == 2` | L | {0, 255} |

**Stop rule:** `TRAINING_BLOCKED` remains `True` until the final promotion gate
(CELL 14) passes. No training, threshold selection, FAUP-Net, or UWACL experiments
may use this dataset until then. Existing PNG and checkpoint results are legacy
evidence only.

In [ ]:
# === CELL 1: Imports and Parameters ===
import os, sys, csv, json, hashlib, zipfile, shutil, tempfile, datetime, uuid
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import nibabel as nib
from PIL import Image
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ── Parameters ────────────────────────────────────────────────────────
DATASET_ROOT         = Path(r"D:\DATA SCIENCE AND ANALYTICS\Dataset\Liver")
RAW_VOLUME_DIR       = DATASET_ROOT / "01_raw_authoritative" / "volumes"
RAW_SEGMENTATION_DIR = DATASET_ROOT / "01_raw_authoritative" / "segmentations"
DOWNLOAD_DIR         = DATASET_ROOT / "01_raw_authoritative" / "nifti_downloads"
STAGING_ROOT         = DATASET_ROOT / "02_staging"
DERIVED_CANONICAL    = DATASET_ROOT / "03_derived_256"

BUILD_ID             = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
BUILD_DIR            = STAGING_ROOT / f"build_{BUILD_ID}"

# ── Safety switches ──────────────────────────────────────────────────
AUDIT_ONLY   = True     # ← CHANGE TO False ONLY after manual review
ALLOW_WRITE  = False     # ← CHANGE TO False ONLY after manual review
PROMOTE      = False     # ← CHANGE TO True ONLY after ALL gates pass

# ── Expected inventory ───────────────────────────────────────────────
EXPECTED_VOLUME_IDS      = set(range(131))
EXPECTED_SEGMENTATION_IDS = set(range(131))

# ── Image profile ────────────────────────────────────────────────────
IMAGE_SIZE      = (256, 256)
RANDOM_SEED     = 42
MASK_RESAMPLE   = "nearest"
CT_WINDOW_CENTER = 40     # HU
CT_WINDOW_WIDTH  = 400    # HU
IMAGE_PROFILE_NAME = "liver_window_256_nearest"

# ── Volume-wise split boundaries (sequential, locked) ────────────────
SPLIT_TRAIN      = set(range(0, 104))     # volumes 0–103
SPLIT_VALIDATION = set(range(104, 117))   # volumes 104–116
SPLIT_TEST       = set(range(117, 131))   # volumes 117–130

TRAINING_BLOCKED = True
DATASET_READY    = False

print(f"BUILD_ID         = {BUILD_ID}")
print(f"BUILD_DIR        = {BUILD_DIR}")
print(f"AUDIT_ONLY       = {AUDIT_ONLY}")
print(f"ALLOW_WRITE      = {ALLOW_WRITE}")
print(f"PROMOTE          = {PROMOTE}")
print(f"TRAINING_BLOCKED = {TRAINING_BLOCKED}")
print(f"IMAGE_PROFILE    = {IMAGE_PROFILE_NAME}")
print(f"CT window        = center {CT_WINDOW_CENTER} HU, width {CT_WINDOW_WIDTH} HU")

In [ ]:
# === CELL 2: Safe I/O and Hash Helpers ===

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def atomic_write_json(path: Path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp")
    with open(tmp, "w") as f:
        json.dump(data, f, indent=2, default=str)
        f.flush()
        os.fsync(f.fileno())
    if path.exists() and sha256_file(path) != sha256_file(tmp):
        raise FileExistsError(f"Hash conflict: {path} already exists with different content")
    tmp.rename(path)

def atomic_write_csv(path: Path, df: pd.DataFrame):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp")
    df.to_csv(tmp, index=False)
    if path.exists() and sha256_file(path) != sha256_file(tmp):
        raise FileExistsError(f"Hash conflict: {path} already exists with different content")
    tmp.rename(path)

def safe_copy_or_extract(source: Path, destination: Path) -> dict:
    source = Path(source)
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    status = {"source": str(source), "destination": str(destination), "action": None}

    if destination.exists():
        src_hash = sha256_file(source)
        dst_hash = sha256_file(destination)
        if src_hash == dst_hash:
            status["action"] = "identical"
            return status
        else:
            status["action"] = "conflict"
            status["src_hash"] = src_hash
            status["dst_hash"] = dst_hash
            raise FileExistsError(f"Hash conflict: {source} vs {destination}")

    if source.suffix == ".zip" and zipfile.is_zipfile(source):
        with zipfile.ZipFile(source) as zf:
            bad = zf.testzip()
            if bad is not None:
                status["action"] = "corrupt"
                status["bad_entry"] = bad
                return status
            zf.extractall(destination.parent)
        status["action"] = "extracted"
    else:
        shutil.copy2(source, destination)
        status["action"] = "copied"
    return status

def validate_png(path: Path, expected_mode: str = "L", expected_size=(256, 256)) -> dict:
    result = {"path": str(path), "valid": False}
    try:
        img = Image.open(path)
        result["mode"] = img.mode
        result["size"] = img.size
        result["dtype_check"] = True
        if img.mode != expected_mode:
            result["error"] = f"mode mismatch: expected {expected_mode}, got {img.mode}"
            return result
        if img.size != expected_size:
            result["error"] = f"size mismatch: expected {expected_size}, got {img.size}"
            return result
        arr = np.array(img)
        unique = set(np.unique(arr).tolist())
        expected_vals = {0, 255} if expected_mode == "L" else None
        if expected_vals and not unique.issubset(expected_vals):
            result["error"] = f"unexpected values: {unique - expected_vals}"
            return result
        result["valid"] = True
    except Exception as e:
        result["error"] = f"{type(e).__name__}: {e}"
    return result

def parse_volume_id(path: Path) -> int:
    name = path.name
    for prefix in ["volume-", "segmentation-"]:
        if name.startswith(prefix):
            rest = name[len(prefix):]
            for ext in [".nii", ".nii.gz", ".zip", ".png"]:
                if rest.endswith(ext):
                    return int(rest[: -len(ext)])
    raise ValueError(f"Cannot parse volume ID from {name}")

# Unit test
print("Cell 2 helpers defined.")
# quick self-test
tmp_dir = BUILD_DIR / "_cell2_test"
tmp_dir.mkdir(parents=True, exist_ok=True)
test_json = tmp_dir / "test.json"
atomic_write_json(test_json, {"test": True})
assert sha256_file(test_json)
print("  atomic_write_json: OK")

# PNG validation test
test_png = tmp_dir / "test_L.png"
Image.fromarray(np.zeros((256, 256), dtype=np.uint8), mode="L").save(test_png)
r = validate_png(test_png, "L", (256, 256))
assert r["valid"], r
print("  validate_png: OK")

shutil.rmtree(tmp_dir)
print("Cell 2: all helpers pass self-test.")

In [ ]:
# === CELL 3: Inventory and Validate Downloads ===

audit_dir = BUILD_DIR / "audits"
audit_dir.mkdir(parents=True, exist_ok=True)

# ── Scan extracted NIfTI volumes ──────────────────────────────────────
vol_files = {}
for p in sorted(RAW_VOLUME_DIR.glob("volume-*.nii")):
    vid = parse_volume_id(p)
    vol_files[vid] = p

# ── Scan extracted NIfTI segmentations ────────────────────────────────
seg_files = {}
for p in sorted(RAW_SEGMENTATION_DIR.glob("segmentation-*.nii")):
    sid = parse_volume_id(p)
    seg_files[sid] = p

# ── Scan ZIP archives ─────────────────────────────────────────────────
zip_dir = DOWNLOAD_DIR / "part2_volumes"
zip_archives = {}
if zip_dir.exists():
    for p in sorted(zip_dir.glob("volume-*.nii.zip")):
        vid = parse_volume_id(p)
        zip_archives[vid] = p

# ── Build inventory ───────────────────────────────────────────────────
all_ids = EXPECTED_VOLUME_IDS | EXPECTED_SEGMENTATION_IDS
inventory_rows = []
for vid in sorted(all_ids):
    row = {
        "volume_id": vid,
        "volume_path": str(vol_files.get(vid, "")),
        "volume_exists": vid in vol_files,
        "segmentation_path": str(seg_files.get(vid, "")),
        "segmentation_exists": vid in seg_files,
        "zip_path": str(zip_archives.get(vid, "")),
        "zip_exists": vid in zip_archives,
    }
    if vid in vol_files:
        row["volume_size_mb"] = round(vol_files[vid].stat().st_size / (1024*1024), 2)
        row["volume_sha256"] = sha256_file(vol_files[vid])
    else:
        row["volume_size_mb"] = None
        row["volume_sha256"] = None
    if vid in seg_files:
        row["seg_size_mb"] = round(seg_files[vid].stat().st_size / (1024*1024), 2)
        row["seg_sha256"] = sha256_file(seg_files[vid])
    else:
        row["seg_size_mb"] = None
        row["seg_sha256"] = None
    inventory_rows.append(row)

inv_df = pd.DataFrame(inventory_rows)
atomic_write_csv(audit_dir / "source_inventory.csv", inv_df)

# ── Validate ZIPs ─────────────────────────────────────────────────────
corrupt_rows = []
for vid, zp in sorted(zip_archives.items()):
    try:
        if not zipfile.is_zipfile(zp):
            corrupt_rows.append({"volume_id": vid, "path": str(zp), "reason": "not a valid ZIP"})
            continue
        with zipfile.ZipFile(zp) as zf:
            bad = zf.testzip()
            if bad is not None:
                corrupt_rows.append({"volume_id": vid, "path": str(zp), "reason": f"corrupt entry: {bad}"})
                continue
            members = zf.namelist()
            expected = f"volume-{vid}.nii"
            if expected not in members:
                corrupt_rows.append({"volume_id": vid, "path": str(zp), "reason": f"unexpected members: {members}"})
    except Exception as e:
        corrupt_rows.append({"volume_id": vid, "path": str(zp), "reason": f"{type(e).__name__}: {e}"})

corrupt_df = pd.DataFrame(corrupt_rows)
atomic_write_csv(audit_dir / "corrupt_downloads.csv", corrupt_df)

# ── Source checksums ──────────────────────────────────────────────────
checksum_rows = []
for vid in sorted(vol_files):
    checksum_rows.append({"id": vid, "type": "volume", "sha256": inv_df.loc[inv_df.volume_id==vid, "volume_sha256"].values[0]})
for sid in sorted(seg_files):
    checksum_rows.append({"id": sid, "type": "segmentation", "sha256": inv_df.loc[inv_df.volume_id==sid, "seg_sha256"].values[0]})
atomic_write_csv(audit_dir / "source_checksums.csv", pd.DataFrame(checksum_rows))

# ── Report ────────────────────────────────────────────────────────────
n_vol = len(vol_files)
n_seg = len(seg_files)
n_zip = len(zip_archives)
missing_vol = sorted(EXPECTED_VOLUME_IDS - set(vol_files.keys()))
missing_seg = sorted(EXPECTED_SEGMENTATION_IDS - set(seg_files.keys()))

print(f"Volumes extracted:   {n_vol}/131")
print(f"Segmentations:       {n_seg}/131")
print(f"ZIP archives:        {n_zip}")
print(f"Missing volumes:     {missing_vol if missing_vol else 'NONE'}")
print(f"Missing segmentations: {missing_seg if missing_seg else 'NONE'}")
print(f"Corrupt ZIPs:        {len(corrupt_rows)}")
if corrupt_rows:
    for r in corrupt_rows:
        print(f"  vol {r['volume_id']}: {r['reason']}")

INV_COMPLETE = (n_vol == 131 and n_seg == 131 and len(missing_vol) == 0)
print(f"\nINV_COMPLETE = {INV_COMPLETE}")
if not INV_COMPLETE:
    print("STOP: Not all 131 volume/segmentation pairs present. Complete downloads first.")

In [ ]:
# === CELL 4: Authoritative NIfTI Pair Validation ===

if not INV_COMPLETE:
    print("SKIPPED: inventory incomplete. Fix downloads and re-run from Cell 3.")
else:
    validation_rows = []
    geometry_rows = []
    affine_mismatches = []
    errors = []

    for vid in sorted(EXPECTED_VOLUME_IDS):
        vp = vol_files[vid]
        sp = seg_files[vid]
        row = {"volume_id": vid}

        try:
            v_img = nib.load(vp)
            s_img = nib.load(sp)
        except Exception as e:
            errors.append({"volume_id": vid, "error": f"nibabel load failed: {e}"})
            row["readable"] = False
            validation_rows.append(row)
            continue

        row["readable"] = True
        v_arr = v_img.get_fdata()
        s_arr = s_img.get_fdata()

        row["image_shape"] = str(v_arr.shape)
        row["seg_shape"] = str(s_arr.shape)
        row["shape_match"] = v_arr.shape == s_arr.shape
        row["image_dtype"] = str(v_arr.dtype)
        row["seg_dtype"] = str(s_arr.dtype)
        row["image_ndim"] = v_arr.ndim
        row["seg_ndim"] = s_img.ndim

        # Voxel spacing
        try:
            row["image_voxel_spacing"] = str(v_img.header.get_zooms())
            row["seg_voxel_spacing"] = str(s_img.header.get_zooms())
        except Exception:
            row["image_voxel_spacing"] = "unknown"
            row["seg_voxel_spacing"] = "unknown"

        # Affine
        row["image_affine"] = str(v_img.affine.tolist())
        row["seg_affine"] = str(s_img.affine.tolist())
        affine_match = np.allclose(v_img.affine, s_img.affine, atol=1e-4)
        row["affine_match"] = affine_match

        # Axis codes
        try:
            row["image_axis_codes"] = str(nib.aff2axcodes(v_img.affine))
            row["seg_axis_codes"] = str(nib.aff2axcodes(s_img.affine))
        except Exception:
            row["image_axis_codes"] = "unknown"
            row["seg_axis_codes"] = "unknown"

        # qform / sform
        row["image_qform_code"] = int(v_img.header.get_qform())
        row["seg_qform_code"] = int(s_img.header.get_qform())

        # Label check
        unique_labels = set(np.unique(s_arr).tolist())
        row["seg_labels"] = str(unique_labels)
        row["labels_valid"] = unique_labels.issubset({0.0, 1.0, 2.0})
        row["has_tumor"] = 2.0 in unique_labels
        row["has_liver"] = 1.0 in unique_labels

        # File hashes
        row["volume_sha256"] = sha256_file(vp)
        row["seg_sha256"] = sha256_file(sp)
        row["image_slices"] = v_arr.shape[2] if v_arr.ndim == 3 else v_arr.shape[0]

        validation_rows.append(row)

        # Geometry record
        geometry_rows.append({
            "volume_id": vid,
            "image_shape": row["image_shape"],
            "seg_shape": row["seg_shape"],
            "shape_match": row["shape_match"],
            "image_voxel_spacing": row["image_voxel_spacing"],
            "seg_voxel_spacing": row["seg_voxel_spacing"],
            "affine_match": affine_match,
            "image_axis_codes": row["image_axis_codes"],
            "seg_axis_codes": row["seg_axis_codes"],
        })

        if not affine_match:
            affine_mismatches.append({
                "volume_id": vid,
                "image_affine": row["image_affine"],
                "seg_affine": row["seg_affine"],
                "image_axis_codes": row["image_axis_codes"],
                "seg_axis_codes": row["seg_axis_codes"],
            })

        if not row["shape_match"]:
            errors.append({"volume_id": vid, "error": f"shape mismatch: vol {v_arr.shape} vs seg {s_arr.shape}"})
        if not row["labels_valid"]:
            errors.append({"volume_id": vid, "error": f"invalid labels: {unique_labels}"})

    # Write audit files
    val_df = pd.DataFrame(validation_rows)
    atomic_write_csv(audit_dir / "nifti_pair_validation.csv", val_df)
    atomic_write_csv(audit_dir / "geometry_report.csv", pd.DataFrame(geometry_rows))
    atomic_write_csv(audit_dir / "affine_mismatches.csv", pd.DataFrame(affine_mismatches))

    # Gate checks
    n_readable = val_df["readable"].sum()
    n_shapes = val_df["shape_match"].sum()
    n_labels = val_df["labels_valid"].sum()
    n_affine_ok = val_df["affine_match"].sum()

    print(f"Readable pairs:      {n_readable}/131")
    print(f"Shape match:         {n_shapes}/131")
    print(f"Labels valid:        {n_labels}/131")
    print(f"Affine match:        {n_affine_ok}/131")
    print(f"Affine mismatches:   {len(affine_mismatches)}")
    if affine_mismatches:
        for m in affine_mismatches:
            print(f"  vol {m['volume_id']}: img axis={m['image_axis_codes']}, seg axis={m['seg_axis_codes']}")
    print(f"Errors:              {len(errors)}")
    if errors:
        for e in errors:
            print(f"  vol {e['volume_id']}: {e['error']}")

    NIFTI_VALID = (n_readable == 131 and n_shapes == 131 and n_labels == 131)
    print(f"\nNIFTI_VALID = {NIFTI_VALID}")
    print(f"Affine mismatches need manual review before proceeding.")

In [ ]:
# === CELL 5: Dry-Run Build Plan ===

if not INV_COMPLETE or not NIFTI_VALID:
    print("SKIPPED: prerequisite gate not met.")
else:
    total_slices = 0
    vol_slice_counts = {}
    for vid in sorted(EXPECTED_VOLUME_IDS):
        vp = vol_files[vid]
        v_img = nib.load(vp)
        n_slices = v_img.shape[2] if v_img.ndim == 3 else v_img.shape[0]
        vol_slice_counts[vid] = n_slices
        total_slices += n_slices

    est_images = total_slices
    est_organ_masks = total_slices
    est_tumor_masks = total_slices
    est_storage_mb = total_slices * 3 * (256 * 256) / (1024 * 1024)  # 3 files per slice

    plan = {
        "build_id": BUILD_ID,
        "n_volumes": 131,
        "total_slices": total_slices,
        "output_file_count": est_images + est_organ_masks + est_tumor_masks,
        "est_storage_mb": round(est_storage_mb, 1),
        "image_size": IMAGE_SIZE,
        "image_profile": IMAGE_PROFILE_NAME,
        "ct_window": {"center": CT_WINDOW_CENTER, "width": CT_WINDOW_WIDTH},
        "mask_resize": MASK_RESAMPLE,
        "mask_mode": "L",
        "mask_values": [0, 255],
        "build_dir": str(BUILD_DIR),
        "per_volume_slices": {str(k): v for k, v in vol_slice_counts.items()},
    }
    atomic_write_json(audit_dir / "dry_run_plan.json", plan)

    print(f"Build ID:            {BUILD_ID}")
    print(f"Volumes:             131")
    print(f"Total slices:        {total_slices:,}")
    print(f"Output files:        {plan['output_file_count']:,}")
    print(f"Est. storage:        {plan['est_storage_mb']:,.1f} MB")
    print(f"Image profile:       {IMAGE_PROFILE_NAME}")
    print(f"CT window:           center={CT_WINDOW_CENTER} HU, width={CT_WINDOW_WIDTH} HU")
    print(f"Mask mode:           L (values 0, 255)")
    print(f"Build directory:     {BUILD_DIR}")
    print(f"\nAUDIT_ONLY = {AUDIT_ONLY}")
    print(f"ALLOW_WRITE = {ALLOW_WRITE}")
    if AUDIT_ONLY or not ALLOW_WRITE:
        print("\nSTOP: AUDIT_ONLY=True or ALLOW_WRITE=False. Edit CELL 1 to enable writes.")

In [ ]:
# === CELL 6: Deterministic Conversion Functions ===

from PIL import ImageFilter

def ct_window_to_uint8(arr: np.ndarray, center: int, width: int) -> np.ndarray:
    """Apply fixed CT window and convert to uint8 [0, 255]."""
    low = center - width / 2
    high = center + width / 2
    arr = np.clip(arr, low, high)
    arr = ((arr - low) / (high - low) * 255).astype(np.uint8)
    return arr

def resize_nearest(arr: np.ndarray, target_size: tuple) -> np.ndarray:
    """Resize 2-D array with nearest-neighbour interpolation."""
    img = Image.fromarray(arr)
    img = img.resize(target_size, Image.NEAREST)
    return np.array(img)

def resize_bilinear(arr: np.ndarray, target_size: tuple) -> np.ndarray:
    """Resize 2-D array with bilinear interpolation (for images only)."""
    img = Image.fromarray(arr)
    img = img.resize(target_size, Image.BILINEAR)
    return np.array(img)

def convert_volume(vol_path: Path, seg_path: Path, build_dir: Path,
                   vol_id: int, window_center: int, window_width: int,
                   image_size: tuple) -> dict:
    """
    Convert one NIfTI pair into PNGs. Returns a manifest dict.
    Same spatial transform applied to image and all masks.
    """
    v_img = nib.load(vol_path)
    s_img = nib.load(seg_path)
    v_arr = v_img.get_fdata().astype(np.float32)
    s_arr = s_img.get_fdata().astype(np.float32)

    n_slices = v_arr.shape[2] if v_arr.ndim == 3 else v_arr.shape[0]
    vol_str = f"v{vol_id:03d}"

    img_dir = build_dir / "images" / vol_str
    organ_dir = build_dir / "organ_masks" / vol_str
    tumor_dir = build_dir / "tumor_masks" / vol_str
    img_dir.mkdir(parents=True, exist_ok=True)
    organ_dir.mkdir(parents=True, exist_ok=True)
    tumor_dir.mkdir(parents=True, exist_ok=True)

    slice_records = []
    total_organ_px = 0
    total_tumor_px = 0

    for si in range(n_slices):
        img_2d = v_arr[:, :, si] if v_arr.ndim == 3 else v_arr[si, :, :]
        seg_2d = s_arr[:, :, si] if s_arr.ndim == 3 else s_arr[si, :, :]

        # Image: apply CT window, resize with bilinear
        img_uint8 = ct_window_to_uint8(img_2d, window_center, window_width)
        img_resized = resize_bilinear(img_uint8, image_size)

        # Organ mask: seg > 0, resize with nearest, save as mode L {0, 255}
        organ_bin = (seg_2d > 0).astype(np.uint8) * 255
        organ_resized = resize_nearest(organ_bin, image_size)

        # Tumor mask: seg == 2, resize with nearest, save as mode L {0, 255}
        tumor_bin = (seg_2d == 2).astype(np.uint8) * 255
        tumor_resized = resize_nearest(tumor_bin, image_size)

        sname = f"s{si:04d}.png"
        img_path = img_dir / sname
        organ_path = organ_dir / sname
        tumor_path = tumor_dir / sname

        Image.fromarray(img_resized, mode="L").save(img_path)
        Image.fromarray(organ_resized, mode="L").save(organ_path)
        Image.fromarray(tumor_resized, mode="L").save(tumor_path)

        organ_px = int(np.count_nonzero(organ_resized))
        tumor_px = int(np.count_nonzero(tumor_resized))
        total_organ_px += organ_px
        total_tumor_px += tumor_px

        slice_records.append({
            "sample_id": f"vol{vol_id:03d}_sli{si:04d}",
            "volume_id": vol_id,
            "slice_index": si,
            "image_path": f"images/{vol_str}/{sname}",
            "organ_mask_path": f"organ_masks/{vol_str}/{sname}",
            "tumor_mask_path": f"tumor_masks/{vol_str}/{sname}",
            "image_exists": True,
            "organ_mask_exists": True,
            "tumor_mask_exists": True,
            "organ_present_256": organ_px > 0,
            "tumor_present_256": tumor_px > 0,
            "organ_pixels_256": organ_px,
            "tumor_pixels_256": tumor_px,
            "source_volume_path": str(vol_path),
            "source_segmentation_path": str(seg_path),
            "source_volume_sha256": sha256_file(vol_path),
            "source_segmentation_sha256": sha256_file(seg_path),
            "preprocessing_profile": IMAGE_PROFILE_NAME,
            "build_id": BUILD_ID,
            "verification_status": "pending_spatial_review",
            "exclusion_reason": "",
        })

    return {
        "volume_id": vol_id,
        "n_slices": n_slices,
        "total_organ_pixels": total_organ_px,
        "total_tumor_pixels": total_tumor_px,
        "slices": slice_records,
        "volume_sha256": sha256_file(vol_path),
        "seg_sha256": sha256_file(seg_path),
    }

print("Cell 6: conversion functions defined.")

In [ ]:
# === CELL 7: Small Dry-Run Conversion ===

DRY_RUN_VOLS = [0, 8, 22, 48, 49, 50]  # ordinary + suspicious

if AUDIT_ONLY or not ALLOW_WRITE:
    print("SKIPPED: AUDIT_ONLY=True or ALLOW_WRITE=False.")
    print("Set AUDIT_ONLY=False and ALLOW_WRITE=True in CELL 1 to run.")
else:
    dry_dir = BUILD_DIR / "dry_run"
    dry_results = []

    for vid in DRY_RUN_VOLS:
        vp = vol_files.get(vid)
        sp = seg_files.get(vid)
        if vp is None or sp is None:
            print(f"  vol {vid}: MISSING, skip")
            continue

        print(f"  vol {vid}: converting...", end=" ", flush=True)
        result = convert_volume(vp, sp, dry_dir, vid, CT_WINDOW_CENTER, CT_WINDOW_WIDTH, IMAGE_SIZE)

        # Validate outputs
        errors = []
        for sr in result["slices"]:
            for key in ["image_path", "organ_mask_path", "tumor_mask_path"]:
                full = dry_dir / sr[key]
                if not full.exists():
                    errors.append(f"missing: {sr[key]}")
                    continue
                vr = validate_png(full, "L", IMAGE_SIZE)
                if not vr["valid"]:
                    errors.append(f"invalid: {sr[key]}: {vr.get('error', 'unknown')}")

            # Containment: tumor subset of organ
            if sr["tumor_present_256"] and not sr["organ_present_256"]:
                errors.append(f"tumor present but organ absent at sli{sr['slice_index']:04d}")
            if sr["tumor_pixels_256"] > sr["organ_pixels_256"]:
                errors.append(f"tumor px ({sr['tumor_pixels_256']}) > organ px ({sr['organ_pixels_256']}) at sli{sr['slice_index']:04d}")

        status = "PASS" if not errors else "FAIL"
        print(f"{status} ({result['n_slices']} slices, organ={result['total_organ_pixels']:,} px, tumor={result['total_tumor_pixels']:,} px)")
        if errors:
            for e in errors:
                print(f"    ERROR: {e}")

        dry_results.append(result)

    n_total = sum(r["n_slices"] for r in dry_results)
    n_tumor = sum(1 for r in dry_results for s in r["slices"] if s["tumor_present_256"])
    print(f"\nDry-run summary: {len(dry_results)} volumes, {n_total} slices, {n_tumor} tumor-positive")
    print(f"Dry-run outputs: {dry_dir}")

In [ ]:
# === CELL 8: Required Spatial Visual Audit ===

REVIEW_VOLUMES = [0, 4, 8, 22, 33, 44, 48, 49, 50]  # ordinary + suspicious

if AUDIT_ONLY or not ALLOW_WRITE:
    print("SKIPPED: AUDIT_ONLY=True or ALLOW_WRITE=False.")
else:
    review_dir = BUILD_DIR / "spatial_reviews"
    review_dir.mkdir(parents=True, exist_ok=True)
    review_records = []

    for vid in REVIEW_VOLUMES:
        vp = vol_files.get(vid)
        sp = seg_files.get(vid)
        if vp is None or sp is None:
            print(f"  vol {vid}: MISSING")
            continue

        v_arr = nib.load(vp).get_fdata().astype(np.float32)
        s_arr = nib.load(sp).get_fdata().astype(np.float32)
        n_slices = v_arr.shape[2] if v_arr.ndim == 3 else v_arr.shape[0]

        # Find key slices
        organ_slices = [si for si in range(n_slices) if np.any(s_arr[:, :, si] > 0)]
        tumor_slices = [si for si in range(n_slices) if np.any(s_arr[:, :, si] == 2)]

        if not organ_slices:
            print(f"  vol {vid}: no organ slices, skip")
            continue

        early_organ = organ_slices[0]
        late_organ = organ_slices[-1]
        mid_idx = len(organ_slices) // 2
        mid_organ = organ_slices[mid_idx]

        # Max tumor burden slice
        if tumor_slices:
            tumor_burdens = [(si, np.sum(s_arr[:, :, si] == 2)) for si in tumor_slices]
            max_tumor_si = max(tumor_burdens, key=lambda x: x[1])[0]
        else:
            max_tumor_si = mid_organ

        # Negative slice
        neg_candidates = [si for si in range(n_slices) if si not in organ_slices]
        neg_slice = neg_candidates[len(neg_candidates)//2] if neg_candidates else None

        # Build review slices list
        review_sis = [early_organ, max_tumor_si, mid_organ, late_organ]
        if neg_slice is not None:
            review_sis.append(neg_slice)
        review_sis = sorted(set(review_sis))

        # Create contact sheet
        n_panels = len(review_sis)
        fig, axes = plt.subplots(1, n_panels, figsize=(4 * n_panels, 4))
        if n_panels == 1:
            axes = [axes]

        for ax, si in zip(axes, review_sis):
            img_2d = v_arr[:, :, si] if v_arr.ndim == 3 else v_arr[si, :, :]
            seg_2d = s_arr[:, :, si] if s_arr.ndim == 3 else s_arr[si, :, :]

            img_uint8 = ct_window_to_uint8(img_2d, CT_WINDOW_CENTER, CT_WINDOW_WIDTH)
            organ_bin = (seg_2d > 0).astype(np.uint8)
            tumor_bin = (seg_2d == 2).astype(np.uint8)

            organ_px = int(np.count_nonzero(resize_nearest(organ_bin * 255, IMAGE_SIZE)) // 255)
            tumor_px = int(np.count_nonzero(resize_nearest(tumor_bin * 255, IMAGE_SIZE)) // 255)

            ax.imshow(img_uint8, cmap="gray")
            # Organ contour (green)
            if np.any(organ_bin):
                organ_resized = resize_nearest(organ_bin, IMAGE_SIZE)
                contours_organ = plt.contour(organ_resized, levels=[0.5], colors="green", linewidths=1)
            # Tumor contour (red)
            if np.any(tumor_bin):
                tumor_resized = resize_nearest(tumor_bin, IMAGE_SIZE)
                contours_tumor = plt.contour(tumor_resized, levels=[0.5], colors="red", linewidths=1.5)

            tag = "NEG" if (neg_slice is not None and si == neg_slice) else ""
            ax.set_title(f"vol{vid:03d} s{si:04d}{tag}\norgan={organ_px} tumor={tumor_px}", fontsize=8)
            ax.axis("off")

        plt.suptitle(f"Spatial Review: volume {vid} | affine_policy=voxel_index_aligned", fontsize=10)
        plt.tight_layout()
        fig_path = review_dir / f"vol{vid:03d}_review.png"
        plt.savefig(fig_path, dpi=150, bbox_inches="tight")
        plt.close()

        # Determine approval status for suspicious volumes
        suspicious = vid in [4, 33, 44, 48, 49, 50]
        status = "needs_review" if suspicious else "approved"

        review_records.append({
            "volume_id": vid,
            "reviewer": "automated_cell8",
            "review_timestamp": datetime.datetime.now().isoformat(),
            "status": status,
            "notes": "suspicious volume, needs manual approval" if suspicious else "ordinary volume, auto-approved",
            "figure_path": str(fig_path),
            "affine_policy": "voxel_index_aligned",
        })
        print(f"  vol {vid}: {fig_path.name} ({n_panels} panels) -> {status}")

    atomic_write_csv(audit_dir / "spatial_audit.csv", pd.DataFrame(review_records))
    print(f"\nSpatial audit saved. {len(review_records)} volumes reviewed.")
    print("Volumes 48, 49, 50 require EXPLICIT MANUAL APPROVAL (affine mismatch).")

In [ ]:
# === CELL 9: Full Build into New Staging Directory ===

if AUDIT_ONLY or not ALLOW_WRITE:
    print("SKIPPED: AUDIT_ONLY=True or ALLOW_WRITE=False.")
else:
    print(f"Building into: {BUILD_DIR}")
    all_slice_records = []
    build_manifest_path = BUILD_DIR / "build_manifest.json"
    resume_manifest = {}
    if build_manifest_path.exists():
        with open(build_manifest_path) as f:
            resume_manifest = {r["volume_id"]: r for r in json.load(f)}

    completed = set()
    for vid in sorted(EXPECTED_VOLUME_IDS):
        vp = vol_files.get(vid)
        sp = seg_files.get(vid)
        if vp is None or sp is None:
            print(f"  vol {vid}: SOURCE MISSING")
            continue

        # Resume check
        if vid in resume_manifest:
            rm = resume_manifest[vid]
            if (rm.get("volume_sha256") == sha256_file(vp) and
                rm.get("seg_sha256") == sha256_file(sp) and
                rm.get("config_hash") == hashlib.sha256(json.dumps({
                    "ct_center": CT_WINDOW_CENTER, "ct_width": CT_WINDOW_WIDTH,
                    "image_size": IMAGE_SIZE, "profile": IMAGE_PROFILE_NAME
                }).encode()).hexdigest() and
                rm.get("status") == "complete"):
                print(f"  vol {vid}: RESUME (hashes match, {rm['n_slices']} slices)")
                # Load slice records from build dir
                vol_slice_dir = BUILD_DIR / "images" / f"v{vid:03d}"
                if vol_slice_dir.exists():
                    n = len(list(vol_slice_dir.glob("s*.png")))
                    # Reconstruct from build dir existence
                    for si in range(n):
                        sname = f"s{si:04d}.png"
                        organ_path = BUILD_DIR / "organ_masks" / f"v{vid:03d}" / sname
                        tumor_path = BUILD_DIR / "tumor_masks" / f"v{vid:03d}" / sname
                        organ_arr = np.array(Image.open(organ_path)) if organ_path.exists() else np.zeros((256,256), dtype=np.uint8)
                        tumor_arr = np.array(Image.open(tumor_path)) if tumor_path.exists() else np.zeros((256,256), dtype=np.uint8)
                        organ_px = int(np.count_nonzero(organ_arr))
                        tumor_px = int(np.count_nonzero(tumor_arr))
                        all_slice_records.append({
                            "sample_id": f"vol{vid:03d}_sli{si:04d}",
                            "volume_id": vid, "slice_index": si,
                            "image_path": f"images/v{vid:03d}/{sname}",
                            "organ_mask_path": f"organ_masks/v{vid:03d}/{sname}",
                            "tumor_mask_path": f"tumor_masks/v{vid:03d}/{sname}",
                            "image_exists": True, "organ_mask_exists": True, "tumor_mask_exists": True,
                            "organ_present_256": organ_px > 0, "tumor_present_256": tumor_px > 0,
                            "organ_pixels_256": organ_px, "tumor_pixels_256": tumor_px,
                            "source_volume_path": str(vp), "source_segmentation_path": str(sp),
                            "source_volume_sha256": sha256_file(vp), "source_segmentation_sha256": sha256_file(sp),
                            "preprocessing_profile": IMAGE_PROFILE_NAME, "build_id": BUILD_ID,
                            "verification_status": "pending_spatial_review", "exclusion_reason": "",
                        })
                completed.add(vid)
                continue

        print(f"  vol {vid}: building...", end=" ", flush=True)
        result = convert_volume(vp, sp, BUILD_DIR, vid, CT_WINDOW_CENTER, CT_WINDOW_WIDTH, IMAGE_SIZE)
        all_slice_records.extend(result["slices"])
        completed.add(vid)

        # Write resume record
        config_hash = hashlib.sha256(json.dumps({
            "ct_center": CT_WINDOW_CENTER, "ct_width": CT_WINDOW_WIDTH,
            "image_size": IMAGE_SIZE, "profile": IMAGE_PROFILE_NAME
        }).encode()).hexdigest()
        resume_manifest[vid] = {
            "volume_id": vid, "status": "complete", "n_slices": result["n_slices"],
            "volume_sha256": sha256_file(vp), "seg_sha256": sha256_file(sp),
            "config_hash": config_hash,
            "total_organ_pixels": result["total_organ_pixels"],
            "total_tumor_pixels": result["total_tumor_pixels"],
        }
        atomic_write_json(build_manifest_path, list(resume_manifest.values()))
        print(f"OK ({result['n_slices']} slices)")

    print(f"\nBuild complete: {len(completed)} volumes, {len(all_slice_records)} slices")
    print(f"Staged at: {BUILD_DIR}

In [ ]:
# === CELL 10: Build the Manifest from Saved Outputs ===

if AUDIT_ONLY or not ALLOW_WRITE:
    print("SKIPPED: AUDIT_ONLY=True or ALLOW_WRITE=False.")
else:
    manifest_rows = []
    for vid in sorted(EXPECTED_VOLUME_IDS):
        vol_str = f"v{vid:03d}"
        img_dir = BUILD_DIR / "images" / vol_str
        if not img_dir.exists():
            continue
        sfiles = sorted(img_dir.glob("s*.png"))
        n_slices = len(sfiles)

        vp = vol_files[vid]
        sp = seg_files[vid]

        for si in range(n_slices):
            sname = f"s{si:04d}.png"
            img_p = BUILD_DIR / "images" / vol_str / sname
            org_p = BUILD_DIR / "organ_masks" / vol_str / sname
            tum_p = BUILD_DIR / "tumor_masks" / vol_str / sname

            # Read from final saved files (the truth)
            org_arr = np.array(Image.open(org_p)) if org_p.exists() else np.zeros(IMAGE_SIZE, dtype=np.uint8)
            tum_arr = np.array(Image.open(tum_p)) if tum_p.exists() else np.zeros(IMAGE_SIZE, dtype=np.uint8)

            organ_px = int(np.count_nonzero(org_arr))
            tumor_px = int(np.count_nonzero(tum_arr))

            manifest_rows.append({
                "sample_id": f"vol{vid:03d}_sli{si:04d}",
                "volume_id": vid,
                "slice_index": si,
                "image_path": f"images/{vol_str}/{sname}",
                "organ_mask_path": f"organ_masks/{vol_str}/{sname}",
                "tumor_mask_path": f"tumor_masks/{vol_str}/{sname}",
                "image_exists": img_p.exists(),
                "organ_mask_exists": org_p.exists(),
                "tumor_mask_exists": tum_p.exists(),
                "organ_present_256": organ_px > 0,
                "tumor_present_256": tumor_px > 0,
                "organ_pixels_256": organ_px,
                "tumor_pixels_256": tumor_px,
                "source_volume_path": str(vp),
                "source_segmentation_path": str(sp),
                "source_volume_sha256": sha256_file(vp),
                "source_segmentation_sha256": sha256_file(sp),
                "preprocessing_profile": IMAGE_PROFILE_NAME,
                "build_id": BUILD_ID,
                "verification_status": "pending_spatial_review",
                "exclusion_reason": "",
            })

    manifest_df = pd.DataFrame(manifest_rows)
    manifest_path = BUILD_DIR / "manifests" / "slice_manifest.csv"
    atomic_write_csv(manifest_path, manifest_df)
    manifest_hash = sha256_file(manifest_path)

    print(f"Manifest: {len(manifest_df)} rows")
    print(f"Volumes:  {manifest_df['volume_id'].nunique()}")
    print(f"Organ present: {manifest_df['organ_present_256'].sum()}")
    print(f"Tumor present: {manifest_df['tumor_present_256'].sum()}")
    print(f"Hash: {manifest_hash}")

In [ ]:
# === CELL 11: Data-Quality and Reconciliation Audit ===

if AUDIT_ONLY or not ALLOW_WRITE:
    print("SKIPPED: AUDIT_ONLY=True or ALLOW_WRITE=False.")
else:
    build_audit = BUILD_DIR / "audits"
    build_audit.mkdir(parents=True, exist_ok=True)

    mdf = manifest_df.copy()
    issues = []

    # 1. Exactly 131 volumes
    n_vols = mdf["volume_id"].nunique()
    vol_ids = set(mdf["volume_id"].unique())
    missing_vols = EXPECTED_VOLUME_IDS - vol_ids
    extra_vols = vol_ids - EXPECTED_VOLUME_IDS
    vols_ok = (n_vols == 131 and not missing_vols and not extra_vols)
    if not vols_ok:
        issues.append({"check": "volume_count", "ok": False, "detail": f"have {n_vols}, missing {missing_vols}, extra {extra_vols}"})

    # 2. Unique sample IDs
    dup_ids = mdf[mdf.duplicated("sample_id", keep=False)]
    dup_ok = len(dup_ids) == 0
    if not dup_ok:
        dup_ids.to_csv(build_audit / "duplicate_keys.csv", index=False)
        issues.append({"check": "unique_sample_ids", "ok": False, "detail": f"{len(dup_ids)} duplicates"})

    # 3. Path existence
    missing_files = []
    for _, row in mdf.iterrows():
        for col in ["image_path", "organ_mask_path", "tumor_mask_path"]:
            full = BUILD_DIR / row[col]
            if not full.exists():
                missing_files.append({"sample_id": row["sample_id"], "missing": col, "path": row[col]})
    files_ok = len(missing_files) == 0
    if not files_ok:
        pd.DataFrame(missing_files).to_csv(build_audit / "missing_data.csv", index=False)
        issues.append({"check": "path_existence", "ok": False, "detail": f"{len(missing_files)} missing"})

    # 4. All outputs 256x256, mode L
    size_issues = []
    for _, row in mdf.head(100).iterrows():  # spot-check first 100
        for col in ["image_path", "organ_mask_path", "tumor_mask_path"]:
            full = BUILD_DIR / row[col]
            if full.exists():
                vr = validate_png(full, "L", IMAGE_SIZE)
                if not vr["valid"]:
                    size_issues.append({"sample_id": row["sample_id"], "file": col, "error": vr.get("error", "")})
    size_ok = len(size_issues) == 0
    if not size_ok:
        issues.append({"check": "output_format", "ok": False, "detail": f"{len(size_issues)} format issues in spot check"})

    # 5. Tumor subset of organ
    containment = mdf[(mdf["tumor_present_256"] == True) & (mdf["organ_present_256"] == False)]
    contain_ok = len(containment) == 0
    if not contain_ok:
        issues.append({"check": "tumor_subset_of_organ", "ok": False, "detail": f"{len(containment)} slices with tumor but no organ"})

    # 6. No pending/needs_review (initially all are pending)
    pending = mdf[mdf["verification_status"].isin(["pending_spatial_review", "needs_review"])]
    pending_ok = len(pending) == 0
    if not pending_ok:
        issues.append({"check": "no_pending_reviews", "ok": False, "detail": f"{len(pending)} pending slices"})

    # 7. Source consistency
    n_source_mix = mdf["source_volume_path"].nunique()
    source_ok = True  # all from NIfTI

    # 8. Label distribution
    label_dist = {
        "total_slices": len(mdf),
        "organ_positive": int(mdf["organ_present_256"].sum()),
        "tumor_positive": int(mdf["tumor_present_256"].sum()),
        "total_organ_pixels": int(mdf["organ_pixels_256"].sum()),
        "total_tumor_pixels": int(mdf["tumor_pixels_256"].sum()),
    }
    atomic_write_json(build_audit / "label_distribution.json", label_dist)

    # Write integrity report
    integrity = {
        "build_id": BUILD_ID,
        "timestamp": datetime.datetime.now().isoformat(),
        "all_pass": all(i.get("ok", True) for i in issues) if issues else True,
        "checks": issues if issues else [{"check": "all", "ok": True, "detail": "passed"}],
        "volume_count": n_vols,
        "total_slices": len(mdf),
    }
    atomic_write_json(build_audit / "integrity_report.json", integrity)

    print(f"Volumes:  {n_vols}/131")
    print(f"Slices:   {len(mdf)}")
    print(f"Issues:   {len(issues)}")
    for i in issues:
        print(f"  [{i['check']}] {i['detail']}")
    AUDIT_PASS = integrity["all_pass"]
    print(f"\nAUDIT_PASS = {AUDIT_PASS}")

In [ ]:
# === CELL 12: Generate Splits from Final Manifest ===

if AUDIT_ONLY or not ALLOW_WRITE:
    print("SKIPPED: AUDIT_ONLY=True or ALLOW_WRITE=False.")
else:
    split_dir = BUILD_DIR / "splits"
    split_dir.mkdir(parents=True, exist_ok=True)

    eligible = mdf[~mdf["verification_status"].isin(["pending_spatial_review", "needs_review", "excluded"])].copy()

    train_df = eligible[eligible["volume_id"].isin(SPLIT_TRAIN)]
    val_df = eligible[eligible["volume_id"].isin(SPLIT_VALIDATION)]
    test_df = eligible[eligible["volume_id"].isin(SPLIT_TEST)]

    # Assertions
    assert len(set(train_df["volume_id"]) & set(val_df["volume_id"])) == 0, "train/val overlap"
    assert len(set(train_df["volume_id"]) & set(test_df["volume_id"])) == 0, "train/test overlap"
    assert len(set(val_df["volume_id"]) & set(test_df["volume_id"])) == 0, "val/test overlap"
    assert len(train_df) + len(val_df) + len(test_df) == len(eligible), "incomplete union"

    # Write volume text files
    for name, ids in [("train_volumes.txt", SPLIT_TRAIN), ("val_volumes.txt", SPLIT_VALIDATION), ("test_volumes.txt", SPLIT_TEST)]:
        with open(split_dir / name, "w") as f:
            for v in sorted(ids):
                f.write(f"{v}\n")

    # Write slice CSVs
    atomic_write_csv(split_dir / "train_slices.csv", train_df)
    atomic_write_csv(split_dir / "val_slices.csv", val_df)
    atomic_write_csv(split_dir / "test_slices.csv", test_df)

    # Split hashes
    split_hashes = {}
    for name in ["train_volumes.txt", "val_volumes.txt", "test_volumes.txt", "train_slices.csv", "val_slices.csv", "test_slices.csv"]:
        split_hashes[name] = sha256_file(split_dir / name)
    atomic_write_json(split_dir / "split_hashes.json", split_hashes)

    print(f"Train: {len(train_df):,} slices (vols 0-103)")
    print(f"Val:   {len(val_df):,} slices (vols 104-116)")
    print(f"Test:  {len(test_df):,} slices (vols 117-130)")
    print(f"Total: {len(train_df)+len(val_df)+len(test_df):,}")
    print(f"Splits written to: {split_dir}")

In [ ]:
# === CELL 13: Version and Provenance Record ===

if AUDIT_ONLY or not ALLOW_WRITE:
    print("SKIPPED: AUDIT_ONLY=True or ALLOW_WRITE=False.")
else:
    version_info = {
        "dataset_name": "LiTS Liver Tumor Segmentation",
        "version": "lits-v0.1.0-unverified",
        "status": "unverified",
        "build_id": BUILD_ID,
        "creation_time": datetime.datetime.now().isoformat(),
        "source": {
            "dataset": "andrewmvd/liver-tumor-segmentation + part-2",
            "n_volumes": len(vol_files),
            "n_segmentations": len(seg_files),
            "volume_ids": sorted(vol_files.keys()),
            "segmentation_ids": sorted(seg_files.keys()),
        },
        "preprocessing": {
            "profile_name": IMAGE_PROFILE_NAME,
            "image_size": list(IMAGE_SIZE),
            "ct_window": {"center": CT_WINDOW_CENTER, "width": CT_WINDOW_WIDTH, "unit": "HU"},
            "image_resize": "bilinear",
            "mask_resize": MASK_RESAMPLE,
            "mask_mode": "L",
            "mask_values": [0, 255],
        },
        "splits": {
            "train_volumes": f"0-103",
            "val_volumes": "104-116",
            "test_volumes": "117-130",
        },
        "counts": {
            "total_slices": len(mdf),
            "organ_positive": int(mdf["organ_present_256"].sum()),
            "tumor_positive": int(mdf["tumor_present_256"].sum()),
            "total_organ_pixels": int(mdf["organ_pixels_256"].sum()),
            "total_tumor_pixels": int(mdf["tumor_pixels_256"].sum()),
        },
        "manifest_hash": manifest_hash,
        "split_hashes": split_hashes,
        "unresolved_warnings": [],
        "manual_review_summary": "Volumes 48-50 require manual approval for affine mismatch.",
    }
    atomic_write_json(BUILD_DIR / "dataset_version.json", version_info)
    print(json.dumps(version_info, indent=2, default=str))

In [ ]:
# === CELL 14: Promotion Gate ===

print("=" * 70)
print("PROMOTION GATE — PASS / FAIL TABLE")
print("=" * 70)

gates = {}

# Gate 1: 131 readable volumes
gates["131_readable_volumes"] = INV_COMPLETE and NIFTI_VALID

# Gate 2: exact ID pairing 0-130
gates["exact_id_pairing_0_130"] = (vol_ids == EXPECTED_VOLUME_IDS) if not AUDIT_ONLY else False

# Gate 3: no shape mismatch
gates["no_shape_mismatch"] = NIFTI_VALID

# Gate 4: all affine anomalies resolved
# Requires manual review of volumes 48-50
spatial_csv = audit_dir / "spatial_audit.csv"
if spatial_csv.exists():
    sa = pd.DataFrame(spatial_csv) if not hasattr(spatial_csv, 'read_text') else pd.read_csv(spatial_csv)
    needs_review = sa[sa["status"] == "needs_review"] if "status" in sa.columns else pd.DataFrame()
    gates["affine_resolved"] = len(needs_review) == 0
else:
    gates["affine_resolved"] = False

# Gate 5: no mixed source
gates["no_mixed_source"] = True  # all from NIfTI by construction

# Gate 6: equal key sets (compare sample_id sets, not path strings)
if not AUDIT_ONLY and ALLOW_WRITE:
    img_ids = set(mdf.loc[mdf["image_exists"], "sample_id"])
    org_ids = set(mdf.loc[mdf["organ_mask_exists"], "sample_id"])
    tum_ids = set(mdf.loc[mdf["tumor_mask_exists"], "sample_id"])
    gates["equal_key_sets"] = (img_ids == org_ids == tum_ids)
else:
    gates["equal_key_sets"] = False

# Gate 7: no missing outputs
if not AUDIT_ONLY:
    gates["no_missing_outputs"] = files_ok
else:
    gates["no_missing_outputs"] = False

# Gate 8: no duplicate keys
if not AUDIT_ONLY:
    gates["no_duplicate_keys"] = dup_ok
else:
    gates["no_duplicate_keys"] = False

# Gate 9: no label mismatch
if not AUDIT_ONLY:
    gates["no_label_mismatch"] = contain_ok
else:
    gates["no_label_mismatch"] = False

# Gate 10: split disjointness
gates["split_disjointness"] = not AUDIT_ONLY and ALLOW_WRITE

# Gate 11: all reports exist
if not AUDIT_ONLY:
    required_reports = ["integrity_report.json", "slice_manifest.csv", "split_hashes.json"]
    gates["all_reports_exist"] = all((BUILD_DIR / "audits" / r).exists() or (BUILD_DIR / r).exists() or (BUILD_DIR / "manifests" / r).exists() or (BUILD_DIR / "splits" / r).exists() for r in required_reports)
else:
    gates["all_reports_exist"] = False

# Gate 12: no pending reviews
gates["no_pending_reviews"] = pending_ok if not AUDIT_ONLY else False

# Print table
all_pass = True
for gate, passed in gates.items():
    status = "PASS" if passed else "FAIL"
    print(f"  {gate:40s} {status}")
    if not passed:
        all_pass = False

print("=" * 70)
print(f"ALL GATES PASS = {all_pass}")

if all_pass and PROMOTE:
    print("\nDATASET_READY=True")
    print("TRAINING_BLOCKED=False")
    print(f"Promoting from {BUILD_DIR} to canonical...")
    # Would promote here — requires explicit PROMOTE=True
else:
    print("\nDATASET_READY=False")
    print("TRAINING_BLOCKED=True")
    if not all_pass:
        print("Failed gates require remediation.")
    if all_pass and not PROMOTE:
        print("All gates pass but PROMOTE=False. Set PROMOTE=True in CELL 11 to promote.")

In [ ]:
# === CELL 15: Training-Loader Contract Test ===

if not (not AUDIT_ONLY and ALLOW_WRITE and all_pass):
    print("SKIPPED: promotion gates not met.")
else:
    print("Training-loader contract test.")
    print("This cell verifies the manifest-driven loader interface.")
    print("")
    print("Loader contract:")
    print("  - Loads train/val from final manifest + splits")
    print("  - Refuses pending/excluded rows")
    print("  - Returns image [1, 256, 256] and binary tumor mask [1, 256, 256]")
    print("  - Uses nearest-neighbour for any mask resize")
    print("  - Never opens test split during training/threshold selection")
    print("  - Deterministic under seed 42")
    print("  - 16-slice overfit test reaches Dice >= 0.80")
    print("")
    print("STATUS: Contract defined. Loader implementation pending.")
    print("Update the project training loader to be manifest-driven before running.")
    print("\nCONTRACT_TEST = False (loader not yet updated)")